## Finding all adverbials from a database

The aim of this notebook is to find all adverbials from the Estonian Reference corpus. This is needed to annotate them with semantic class using both rule based methods and LLMs. The code extracts data from Katrin Tsepelina's database [v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db](https://github.com/estnltk/syntax_experiments/tree/verb_templates/workflows/001_verb_transactions/v33) with data extracted from the Estonian Reference corpus.

This code creates a 2 new tables in the database
1. **spatial_obl** with nominal adverbials aka obliques in spatial cases (form + lemma + feats), their head verb (verb+compund) and sentences the obliques came from.
2. **advmod** with adverbs (form + lemma), their head verbs (verb+compound) and sentences the adverbs came from

The sentences are taken from another database and added to this one based on their sentence id.

Ner and timex tags are taken from another database and added to this one based on their sentence id and location in sentence.

In [1]:
#imports
import sqlite3
import pandas as pd

In [23]:
# database file path
DB_FILE = "../../drive_data/v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db"
NER_TIMEX_DB = "../../drive_data/v33_ner_timex_20250922-094340.db"
SENTENCES_DB = "../../drive_data/v33_koondkorpus_sentences_sentences_20250220-130121.db"

OBL_TABLE = "spatial_obl"
ADVMOD_TABLE = "advmod"

# column for final new tag
TAG_COL = "tag"
NEW_EKILEX_COL = "new_ekilex_tag"

In [3]:
# connecting with database
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

### Create new table spatial_obl

In [4]:
cursor.execute(f"DROP TABLE if exists {OBL_TABLE}")

In [5]:
#searches for obliques in spatial cases + head verb + sentence id
query = (f"CREATE TABLE {OBL_TABLE} AS " 
         f"SELECT transaction_row.id, sentence_id, transaction_row.head_id, transaction_row.loc as row_loc, verb, verb_compound, transaction_row.feats, pos, lemma, transaction_row.form, status, ekilex_tag " 
         f"FROM `transaction_row` JOIN `transaction_head` ON transaction_head.id = transaction_row.head_id WHERE transaction_row.deprel = 'obl' "
        f"AND (transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ? OR transaction_row.feats LIKE ?)")
#print(query)

cursor.execute(query, ('%adit,%', '%ill,%', '%in,%', '%el,%', '%all,%', '%ad,%', '%abl,%'))

### Create new table advmod

In [6]:
cursor.execute(f"DROP TABLE if exists {ADVMOD_TABLE}")

In [7]:
#searches for obliques in spatial cases + head verb + sentence id
query = (f"CREATE TABLE {ADVMOD_TABLE} AS " 
         f"SELECT transaction_row.id, transaction_row.head_id, transaction_row.form, transaction_row.lemma, pos, verb, verb_compound, sentence_id " 
         f"FROM `transaction_row` JOIN `transaction_head` ON transaction_head.id = transaction_row.head_id WHERE transaction_row.deprel = 'advmod' ")
cursor.execute(query)

### Add sentences to tables

In [6]:
def column_exists(cursor, table, column):
    cursor.execute(f"PRAGMA table_info({table})")
    return any(row[1] == column for row in cursor.fetchall())

In [7]:
def sentences_to_table(input_table, sentence_db, target_db):
    
    # Connect to both databases
    conn1 = sqlite3.connect(sentence_db)  # Source database
    conn2 = sqlite3.connect(target_db)  # Target database

    cursor1 = conn1.cursor()
    cursor2 = conn2.cursor()
    
    
    # Step 1: Retrieve sentences from database1
    cursor1.execute("SELECT id, text FROM sentences")
    sentences = cursor1.fetchall()  # List of (sentence_id, sentence)

    # Step 2: add new column to database2 table
    if not column_exists(cursor2, input_table, "sentence"):
        cursor2.execute("ALTER TABLE " + input_table +  " ADD COLUMN sentence TEXT")

    # Step 3: Create a temporary table
    cursor2.execute("CREATE TEMP TABLE temp_sentence (id INT PRIMARY KEY, sentence TEXT)")

    # Step 4: Insert all values into the temp table
    cursor2.executemany("INSERT INTO temp_sentence (id, sentence) VALUES (?, ?)", sentences)

    # Step 3: Perform a fast join-based update
    query = f"""
        UPDATE {input_table}
        SET sentence = (
            SELECT sentence
            FROM temp_sentence
            WHERE temp_sentence.id = {input_table}.sentence_id
            LIMIT 1
         )
        WHERE EXISTS (
            SELECT 1
            FROM temp_sentence
            WHERE temp_sentence.id = {input_table}.sentence_id
        )
    """

    cursor2.execute(query)

    conn2.commit()
    conn1.close()
    conn2.close()

In [8]:
sentences_to_table(OBL_TABLE, SENTENCES_DB, DB_FILE)

In [11]:
sentences_to_table(ADVMOD_TABLE, SENTENCES_DB, DB_FILE)

In [9]:
conn.close()

### Add NER and TIMEX tags

In [10]:
# Connect to both databases
conn_source = sqlite3.connect(NER_TIMEX_DB)  # Source database
conn_target = sqlite3.connect(DB_FILE)  # Target database

cursor_source = conn_source.cursor()
cursor_target = conn_target.cursor()

In [11]:
def ner_timex_to_table(input_table, cursor1, cursor2, conn1, conn2):
    # Step 1: Retrieve data from database1
    cursor1.execute("SELECT sentence_id, ner_tag, loc FROM ner")
    ners = cursor1.fetchall()  # List of (sentence_id, ner_tag, loc)

    # Step 2: add new column to database2 table
    if not column_exists(cursor2, input_table, "ner_tag"):
        cursor2.execute("ALTER TABLE " + input_table +  " ADD COLUMN ner_tag TEXT")

    # Step 3: Create a temporary table
    cursor2.execute("CREATE TEMP TABLE temp_ner (sentence_id INT, ner_tag TEXT, loc INT)")

    # Step 4: Insert all values into the temp table
    cursor2.executemany("INSERT INTO temp_ner (sentence_id, ner_tag, loc) VALUES (?, ?, ?)", ners)

    cursor2.execute(f"""
        UPDATE {input_table}
        SET ner_tag = temp_ner.ner_tag
        FROM temp_ner
        WHERE {input_table}.sentence_id = temp_ner.sentence_id and {input_table}.row_loc=temp_ner.loc;
    """)
    
    conn2.commit()
    
    
    
    # Step 1: Retrieve data from database1
    cursor1.execute("SELECT sentence_id, timex_type, loc FROM timex")
    timexes = cursor1.fetchall()  # List of (sentence_id, timex_type, loc)

    # Step 2: add new column to database2 table
    if not column_exists(cursor2, input_table, "timex_tag"):
        cursor2.execute("ALTER TABLE " + input_table +  " ADD COLUMN timex_tag TEXT")

    # Step 3: Create a temporary table
    cursor2.execute("CREATE TEMP TABLE temp_timex (sentence_id INT, timex_type TEXT, loc INT)")

    # Step 4: Insert all values into the temp table
    cursor2.executemany("INSERT INTO temp_timex (sentence_id, timex_type, loc) VALUES (?, ?, ?)", timexes)

    # Step 3: Perform a fast join-based update
    cursor2.execute(f"""
        UPDATE {input_table}
        SET timex_tag = temp_timex.timex_type
        FROM temp_timex
        WHERE {input_table}.sentence_id = temp_timex.sentence_id and {input_table}.row_loc=temp_timex.loc;
    """)
    
    conn2.commit()

    conn1.close()
    conn2.close()

In [12]:
ner_timex_to_table(OBL_TABLE, cursor_source, cursor_target, conn_source, conn_target)

In [13]:
conn_source.close()
conn_target.close()

### Create new_ekilex_tag based on ekilex tags
New tags:

- time/location/event -> TLE
- STATE
- ALIVE


In [24]:
# connecting with database
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

In [25]:

if not column_exists(cursor, OBL_TABLE, NEW_EKILEX_COL):
    cursor.execute("ALTER TABLE " + OBL_TABLE +  f" ADD COLUMN {NEW_EKILEX_COL} TEXT")

# Update values 
cursor.execute(f"""
UPDATE {OBL_TABLE}
SET {NEW_EKILEX_COL} = CASE
    -- STATE
    WHEN ekilex_tag = 'state' THEN 'STATE'
    
    -- ALIVE
    WHEN ekilex_tag = 'alive' THEN 'ALIVE'
    
    -- TLE
    WHEN ekilex_tag IN ('time', 'location', 'event') THEN 'TLE'
    
    ELSE NULL
END
""")

conn.commit()
conn.close()

### Create final tag based on ekilex, ner and timex tags

New tags:

- time/location/event -> TLE
- STATE
- alive/per -> ALIVE
- ORG

In [14]:
# connecting with database
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

In [15]:

if not column_exists(cursor, OBL_TABLE, TAG_COL):
    cursor.execute("ALTER TABLE " + OBL_TABLE +  f" ADD COLUMN {TAG_COL} TEXT")

# Update values 
cursor.execute(f"""
UPDATE {OBL_TABLE}
SET {TAG_COL} = CASE
    -- STATE
    WHEN ekilex_tag = 'state' THEN 'STATE'
    
    -- ORG
    WHEN ner_tag = 'ORG' THEN 'ORG'
    
    -- ALIVE
    WHEN ekilex_tag = 'alive' OR ner_tag = 'PER' THEN 'ALIVE'
    
    -- TLE
    WHEN ekilex_tag IN ('time', 'location', 'event')
         OR ner_tag = 'LOC'
         OR timex_tag IS NOT NULL THEN 'TLE'
    
    ELSE NULL
END
""")

conn.commit()
conn.close()

### Separate cases
Separating the case tag from the larger feats value and adding it to the database table

In [16]:
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

cursor.execute(f"ALTER TABLE {OBL_TABLE} ADD COLUMN morph_case TEXT")

In [17]:
# separate cases
cases = ['adit', 'ill', 'in', 'el', 'all', 'ad', 'abl']

# Fetch feats column
cursor.execute(f"SELECT id, feats FROM {OBL_TABLE}")
rows = cursor.fetchall()

# find case and separate into separate column
updates = []
for rowid, feats in rows:
    if feats:
        for case in cases:
            if case in feats.split(","):
                case_value = case
        updates.append((case_value, rowid))

In [18]:
# add case info to main table

# Step 1: Create a temporary table
cursor.execute("CREATE TEMP TABLE temp_case (id INT PRIMARY KEY, morph_case TEXT)")

# Step 2: Insert all values into the temp table
cursor.executemany("INSERT INTO temp_case (morph_case, id) VALUES (?, ?)", updates)

# Step 3: Perform a fast join-based update
cursor.execute(f"""
    UPDATE {OBL_TABLE}
    SET morph_case = (SELECT morph_case FROM temp_case WHERE temp_case.id = spatial_obl.id)
""")

# Commit changes and close connection
conn.commit()
conn.close()

## kontroll

In [26]:

# connecting with database
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

In [27]:
query = f"SELECT * FROM spatial_obl where tag is not null limit 20"

res = pd.read_sql(query, conn)
res

,id,sentence_id,head_id,row_loc,verb,verb_compound,feats,pos,lemma,form,status,ekilex_tag,sentence,ner_tag,timex_tag,tag,morph_case,new_ekilex_tag
0,1,3,2,3,toimuma,,"com,in,sg",S,lõpp,lõpus,,None,Kaheksakümnendate aastate lõpus toimus Türi 1.,None,DATE,TLE,in,None
1,59,25,33,12,minema,peale,"all,com,sg",S,rahvas,rahvale,,alive,"Tegutsesin siis diskorina ja nägin , et eurobi...",None,None,ALIVE,all,ALIVE
2,119,50,74,5,saama,,"com,el,sg",S,toidupood,toidupoest,,location,"Kui sa ei saa toidupoest soovitut , on asi pär...",None,None,TLE,el,TLE
3,129,54,84,3,loobuma,,"ad,com,sg",S,aeg,ajal,,None,Olen viimasel ajal juhutöödest loobunud ja kir...,None,DATE,TLE,ad,None
4,131,54,85,9,kirjutama,,"all,com,pl",S,bänd,bändidele,,alive,Olen viimasel ajal juhutöödest loobunud ja kir...,None,None,ALIVE,all,ALIVE
5,148,63,96,5,saama,,"com,el,pl",S,inimene,inimestest,,alive,"Ma ei saa aru inimestest , kes üldse napsu ei ...",None,None,ALIVE,el,ALIVE
6,155,63,98,18,jooma,,"com,el,sg",S,hommik,hommikust,,None,"Ma ei saa aru inimestest , kes üldse napsu ei ...",None,TIME,TLE,el,None
7,170,68,109,13,tahtma,,"com,el,sg",S,pidu,peost,,event,Narkootikumidest ja vahel ka napsust on kerge ...,None,None,TLE,el,TLE
8,196,83,135,5,tulema,,"all,com,sg",S,kontsert,kontserdile,,event,"Inimest , kes tuleb kontserdile , näeb sind es...",None,None,TLE,all,TLE
9,205,84,142,14,helistama,,"com,el,sg",S,hommik,hommikust,,None,"Mõned tüdrukud muutusid lausa tüütuks , uurisi...",None,TIME,TLE,el,None


In [21]:
query = f"SELECT * FROM spatial_obl where tag='TLE' limit 10"

res = pd.read_sql(query, conn)
res

,id,sentence_id,head_id,row_loc,verb,verb_compound,feats,pos,lemma,form,status,ekilex_tag,sentence,ner_tag,timex_tag,tag,morph_case
0,1,3,2,3,toimuma,,"com,in,sg",S,lõpp,lõpus,,None,Kaheksakümnendate aastate lõpus toimus Türi 1.,None,DATE,TLE,in
1,119,50,74,5,saama,,"com,el,sg",S,toidupood,toidupoest,,location,"Kui sa ei saa toidupoest soovitut , on asi pär...",None,None,TLE,el
2,129,54,84,3,loobuma,,"ad,com,sg",S,aeg,ajal,,None,Olen viimasel ajal juhutöödest loobunud ja kir...,None,DATE,TLE,ad
3,155,63,98,18,jooma,,"com,el,sg",S,hommik,hommikust,,None,"Ma ei saa aru inimestest , kes üldse napsu ei ...",None,TIME,TLE,el
4,170,68,109,13,tahtma,,"com,el,sg",S,pidu,peost,,event,Narkootikumidest ja vahel ka napsust on kerge ...,None,None,TLE,el
5,196,83,135,5,tulema,,"all,com,sg",S,kontsert,kontserdile,,event,"Inimest , kes tuleb kontserdile , näeb sind es...",None,None,TLE,all
6,205,84,142,14,helistama,,"com,el,sg",S,hommik,hommikust,,None,"Mõned tüdrukud muutusid lausa tüütuks , uurisi...",None,TIME,TLE,el
7,221,92,157,2,täitma,,"ad,com,sg",S,aasta,aastal,,None,"Sel aastal täitsime oma missiooni , mille eesm...",None,DATE,TLE,ad
8,248,100,172,4,lõpetama,,"com,in,sg",S,pedagoogikaülikool,pedagoogikaülikoolis,,location,Sa oled lõpetanud pedagoogikaülikoolis orkestr...,None,None,TLE,in
9,281,110,190,13,kavatsema,,"ad,com,sg",S,kevad,kevadel,,time,Mul tekkis viis aastat tagasi tõsine huvi lenn...,None,DATE,TLE,ad


In [28]:
conn.close()